# 13: Shift (BRFSS temporal and geographic)

Design decisions
- One fixed model per source, trained like the frozen heart model: frozen XGBoost settings, seed 42, Youden threshold on 5-fold out-of-fold scores of the source training rows. Each model has its own prior (mixture with the frozen lambda, candidate sets of 1,000 source training patients) and its own box certificates (the source records of calibration split 1). The model hash is recorded when the model is first fitted and checked before every target evaluation; refitting must reproduce it.
- Temporal: source BRFSS 2015 with the 19 predictors of H (asked in every year; no fruit or vegetables), targets 2017, 2019, 2021 and 2023. The frozen splits apply to the rebuilt 2015 table, which equals the public table row by row (checked below).
- Geographic: source three census regions of 2015, target the fourth (four models, all 21 predictors, frozen splits restricted to the source regions). The five states with the most 2015 records are targets of the model trained without their region. Guam and Puerto Rico belong to no census region and are left out.
- Per target, 1,000 explained patients are drawn at random (500 per state and for the no-shift reference, the source test split). The other target records check claims.
- Transport: release with the source certificates at the frozen targets (alpha_0 = 0.05, alpha_1 = 0.70). Claims are checked on target labels: the selective error of the explained patients, and box risks on the other target records (a claim is refuted when the Bonferroni-corrected exact lower 95% limit is above its target).
- Re-certification: labelled target samples of 500, 1,000, 2,000, 5,000 and 10,000 records (20 draws each) replace the source certificates; model, candidates and prior stay fixed. At least 1,000 target records are left to check claims, so larger sizes are skipped for small targets. The no-shift reference gets the same curve, which separates the sample-size effect from the shift.
- Diagnostics: a domain classifier (gradient boosting, 2-fold cross-fitting, up to 50,000 records per domain) gives the overall AUROC and, for every box released with source certificates, the within-box AUROC and the b/a estimate of Eq. 12 (ratio of the 95% and 5% quantiles of the estimated density ratio in the box). Black-box shift estimation (Lipton et al. 2018) gives the target prevalence that pure label shift would imply.
- Site (UCI) and measurement (NHANES) shift need notebook 03 and follow in notebook 13b.
- Every finished piece is saved on Drive (models, candidate sets every 500 new cells, each target's results), so an interrupted run resumes where it stopped. Expected run time on Colab CPU: about 2 to 3 hours, almost all of it candidate sets (about 18,000 patients).

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time, zlib
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public, splits
from scax.data import brfss_raw as br
from scax.certify.clopper_pearson import cp_lower, cp_upper
from scax.experiments.main_results import FrozenTask
from scax.models.export import Ensemble
try:
    from scax.experiments import shift as sh
    from scax.experiments.model_ablations import exported_vs_native, fit_variant
except ImportError:
    raise RuntimeError('The stage 9 source is not loaded. Run the stage 9 batch, then Runtime > Disconnect and delete runtime, then run again.')

SEED = 42
seed_everything(SEED)
TASK = 'brfss2015_heart'
KIND = {0: 'negative outcome', 1: 'enrichment'}
REC05 = json.load(open(f'{paths.RESULTS}/05_frozen.json'))
FT = FrozenTask(TASK, REPO_DIR, DRIVE_ROOT, REC05['frozen_config_sha256'][TASK])
REC02 = json.load(open(f'{paths.RESULTS}/02_brfss_rebuild.json'))
assert REC02['gate_G0']['passed'] and REC02['gate_G0']['row_by_row_equal'], 'gate G0 of notebook 02 did not pass'
H = list(REC02['harmonised_set_H'])
FULL = list(FT.ens.feature_names)
YEARS = list(REC02['years_used'])
TAB = {}
for year in YEARS:
    f = paths.DERIVED / 'brfss' / f'brfss_{year}.parquet'
    assert sha256_file(f) == REC02['parquet_sha256'][str(year)], f'{f.name} differs from the file notebook 02 wrote'
    TAB[year] = pd.read_parquet(f)
code = np.load(paths.DERIVED / 'splits' / f'{TASK}.npy')
assert splits.summary(code)['sha256'] == FT.cfg['split_sha256'], 'split differs from the frozen one'
src15 = TAB[2015]
X_pub, y_pub = public.LOADERS[TASK](paths.RAW)
assert len(src15) == len(code) and np.array_equal(src15[FULL].to_numpy(float), X_pub[FULL].to_numpy(float)) \
    and np.array_equal(src15[br.TARGET].to_numpy(int), y_pub), 'the rebuilt 2015 table must equal the public table row by row'
del X_pub, y_pub

PARAMS = FT.cfg['model']['params']
CAND = FT.cfg['candidates']
ALPHA = {0: FT.targets['alpha_0'], 1: FT.targets['alpha_1']}
DELTA1 = FT.targets['delta_1']
N_PRIOR, N_EXPLAIN, N_REF, N_STATE = 1000, 1000, 500, 500
SIZES, DRAWS, MIN_EVAL = [500, 1000, 2000, 5000, 10000], 20, 1000
SHIFT_DIR = paths.DERIVED / 'shift'
for d in ('models', 'results'):
    (SHIFT_DIR / d).mkdir(parents=True, exist_ok=True)
print('frozen configuration', FT.cfg_sha[:16], '| settings', PARAMS, '| alpha', ALPHA, '| delta_1', DELTA1)
print(f'H: {len(H)} predictors | years {YEARS} | records per year', {y: len(t) for y, t in TAB.items()})

In [ ]:
MODELS, RESULTS = {}, []
LOG = lambda m: print(m, flush=True)


def seed_of(*parts):
    return zlib.crc32('|'.join(map(str, parts)).encode()) % (2 ** 31)


def build_model(name, X_tr, y_tr, names, X_prior, X_cal, y_cal):
    """One fixed model (the fit is deterministic), checked against its saved copy; then its prior and certificates."""
    t0 = time.time()
    model, ens, pred = fit_variant('xgboost', X_tr, y_tr, PARAMS, SEED, names)
    f = SHIFT_DIR / 'models' / f'{name}__model.json'
    if f.exists():
        assert Ensemble.load(f).hash() == ens.hash(), f'{name}: refitting does not reproduce the saved model'
    else:
        ens.save(f)
    disagree = exported_vs_native(model, 'xgboost', pred, X_cal)
    assert disagree == 0, f'{name}: exported and native models disagree on {disagree} rows'
    sm = sh.ShiftModel(name, pred, CAND['M'], CAND['order_seed'], CAND['node_budget'], FT.lam, ALPHA, DELTA1,
                       cache_path=SHIFT_DIR / 'models' / f'{name}__candidates.pkl')
    sm.set_prior(X_prior, log=LOG)
    sm.set_calibration(X_cal, y_cal)
    MODELS[name] = {'model_sha256': ens.hash(), 'predictors': names, 'training_records': int(len(y_tr)),
                    'training_prevalence': float(np.mean(y_tr)), 'calibration_records': int(len(y_cal)),
                    'prior_patients': int(len(X_prior)), 'prior_boxes': len(sm.counts),
                    'thresholds': int(sum(len(t) for t in pred.grid.thresholds)), 'tau': float(ens.tau),
                    'exported_vs_native_disagreements': disagree}
    print(f"{name}: {len(y_tr):,} training records, {MODELS[name]['thresholds']} thresholds, {len(y_cal):,} calibration "
          f"records, hash {ens.hash()[:12]}, {time.time() - t0:.0f}s", flush=True)
    return sm


def run_target(sm, setting, target, role, X_t, y_t, n_explain):
    """Transport, re-certification and diagnostics for one target; saved on Drive when finished."""
    res_file = SHIFT_DIR / 'results' / f"{sm.name}__{target.replace(' ', '_').replace(':', '')}.json"
    if res_file.exists():
        r = json.load(open(res_file))
        assert r['model_sha256'] == sm.model_hash, f'{res_file.name} was computed with a different model'
        RESULTS.append(r); print(f'{sm.name} | {target}: loaded from Drive', flush=True)
        return r
    t0 = time.time()
    assert sm.model_hash == MODELS[sm.name]['model_sha256']
    sm.check_hash()
    s = seed_of(sm.name, target)
    rng = np.random.default_rng(s)
    exp = np.sort(rng.choice(len(y_t), n_explain, replace=False))
    rest = np.setdiff1d(np.arange(len(y_t)), exp)
    te = sh.TargetEval(sm, X_t[exp], y_t[exp], X_t[rest], y_t[rest], log=LOG)
    met = te.transport()
    recert = te.recertify(SIZES, DRAWS, s, MIN_EVAL)
    i_s, i_t, p_s, p_t, dom = sh.domain_scores(sm.X_cal, X_t[rest], s)
    ids = np.unique(te.rel_transport[te.rel_transport >= 0])
    boxes = sh.box_diagnostics(te, ids, sm.C_cal[i_s], te.C_rest[i_t], p_s, p_t)
    info = {**sh.model_summary(sm.pred, X_t, y_t), 'explained': int(n_explain), 'checking_records': int(len(rest)),
            'source_prevalence': float(sm.y_cal.mean()),
            'bbse_prevalence': sh.bbse_prevalence(sm.yhat_cal, sm.y_cal, sm.pred.predict(X_t[rest])),
            'domain_auroc': dom, 'candidate_boxes': len(te.keys), 'seed': int(s)}
    r = {'setting': setting, 'model': sm.name, 'target': target, 'role': role, 'model_sha256': sm.model_hash, 'info': info,
         'transport': [{'kind': k, **met[k]} for k in (0, 1)], 'recert': recert,
         'boxes': boxes.round(6).to_dict('records'), 'seconds': time.time() - t0}
    write_json(res_file, r)
    RESULTS.append(r)
    print(f"{sm.name} | {target}: AUROC {info['auroc']:.3f}, prevalence {info['prevalence']:.1%}, domain AUROC {dom:.3f}; "
          f"source certificates explain {met[0]['coverage']:.1%} / {met[1]['coverage']:.1%} (negative / enrichment), "
          f"errors {met[0]['error_rate']:.1%} / {met[1]['error_rate']:.1%}, claims refuted on target "
          f"{met[0]['refuted_share']:.1%} / {met[1]['refuted_share']:.1%}; {time.time() - t0:.0f}s", flush=True)
    return r

## Temporal shift: one model trained on 2015 with the 19 predictors of H

In [ ]:
XH = src15[H].to_numpy(float)
y15 = src15[br.TARGET].to_numpy(int)
tr, cal1, test = (splits.indices(code, s) for s in ('train', 'cal1', 'test'))
prior = np.sort(np.random.default_rng(seed_of('prior', 'temporal')).choice(tr, N_PRIOR, replace=False))
sm = build_model('temporal_2015_H', XH[tr], y15[tr], H, XH[prior], XH[cal1], y15[cal1])
run_target(sm, 'temporal', '2015 test split', 'reference', XH[test], y15[test], N_REF)
for year in [y for y in YEARS if y != 2015]:
    t = TAB[year]
    run_target(sm, 'temporal', str(year), 'target', t[H].to_numpy(float), t[br.TARGET].to_numpy(int), N_EXPLAIN)
del sm

## Geographic shift: three census regions of 2015 to the fourth, and to the five states with most records

In [ ]:
XF = src15[FULL].to_numpy(float)
STATE = src15['_STATE'].to_numpy(dtype=float)
REG = sh.region_of(STATE)
HAS_REGION = np.array([r is not None for r in REG])
STATES = sh.largest_states(STATE[HAS_REGION], 5)
REGIONS = pd.DataFrame([{'region': r, 'records': int((REG == r).sum()), 'heart_disease': float(y15[REG == r].mean())}
                        for r in sh.CENSUS_REGIONS])
print(REGIONS.round(4).to_string(index=False))
print(f'records without a census region (territories, other codes): {int((~HAS_REGION).sum()):,}')
print('five states with most records:', [(sh.STATE_NAMES[f], int((STATE == f).sum()), sh.region_of([f])[0]) for f in STATES])
for region in sh.CENSUS_REGIONS:
    in_t = REG == region
    in_s = HAS_REGION & ~in_t
    s_tr = np.flatnonzero(in_s & (code <= 1))                  # training portion of the source regions
    s_cal = np.flatnonzero(in_s & (code == 2))                 # calibration split 1 of the source regions
    s_test = np.flatnonzero(in_s & (code == 4))                # test split of the source regions
    prior = np.sort(np.random.default_rng(seed_of('prior', region)).choice(s_tr, N_PRIOR, replace=False))
    sm = build_model(f'geographic_without_{region}', XF[s_tr], y15[s_tr], FULL, XF[prior], XF[s_cal], y15[s_cal])
    run_target(sm, 'geographic', f'{region} held out: source test split', 'reference', XF[s_test], y15[s_test], N_REF)
    t = np.flatnonzero(in_t)
    run_target(sm, 'geographic', region, 'target', XF[t], y15[t], N_EXPLAIN)
    for f in STATES:
        if sh.region_of([f])[0] == region:
            t = np.flatnonzero(STATE == f)
            run_target(sm, 'state', sh.STATE_NAMES[f], 'target', XF[t], y15[t], N_STATE)
    del sm

## Results

In [ ]:
ID = ('setting', 'model', 'target', 'role')
TGT = pd.DataFrame([{**{k: r[k] for k in ID}, **r['info'], 'seconds': r['seconds']} for r in RESULTS])
TRN = pd.DataFrame([{**{k: r[k] for k in ID}, **row} for r in RESULTS for row in r['transport']])
DRW = pd.DataFrame([{**{k: r[k] for k in ID}, **row} for r in RESULTS for row in r['recert']])
REC = pd.concat([sh.summarise_recert(r['recert']).assign(**{k: r[k] for k in ID}) for r in RESULTS if r['recert']],
                ignore_index=True)
BOX = pd.DataFrame([{**{k: r[k] for k in ID}, **row} for r in RESULTS for row in r['boxes']])
TRN = TRN.merge(TGT[['model', 'target', 'explained']], on=['model', 'target'])
TRN['coverage_lo'] = [float(cp_lower(a, n, 0.05)) for a, n in zip(TRN.accepted, TRN.explained)]
TRN['coverage_hi'] = [float(cp_upper(a, n, 0.05)) for a, n in zip(TRN.accepted, TRN.explained)]
ref = TRN[TRN.role == 'reference'][['model', 'kind', 'coverage', 'error_rate']].rename(
    columns={'coverage': 'no_shift_coverage', 'error_rate': 'no_shift_error_rate'})
TRN = TRN.merge(ref, on=['model', 'kind'], how='left')
REC['kind_name'] = REC.kind.map(KIND); TRN['kind_name'] = TRN.kind.map(KIND)
pd.set_option('display.width', 250)
print(TGT[['setting', 'target', 'records', 'prevalence', 'auroc', 'sensitivity', 'specificity', 'source_prevalence',
           'bbse_prevalence', 'domain_auroc', 'candidate_boxes']].round(4).to_string(index=False))
for k in (0, 1):
    print(f'\nSource certificates carried to the target, {KIND[k]} claims (target {ALPHA[k]:.2f})')
    print(TRN[TRN.kind == k][['setting', 'target', 'explained', 'coverage', 'coverage_lo', 'coverage_hi', 'no_shift_coverage',
                              'error_rate', 'error_upper_95', 'refuted_share', 'above_target_share',
                              'certificate_refuted_share', 'mean_size']].round(4).to_string(index=False))
for k in (0, 1):
    print(f'\nRe-certification with labelled target records, {KIND[k]} claims: share of explained patients '
          f'(mean of {DRAWS} draws)')
    piv = REC[REC.kind == k].pivot_table(index=['setting', 'target'], columns='m', values='coverage', sort=False)
    print(piv.round(3).to_string())
    bad = REC[(REC.kind == k) & (REC.draws_with_refuted_claims > 0)]
    print('draws with a claim refuted on target records:',
          'none' if bad.empty else bad[['target', 'm', 'draws_with_refuted_claims']].to_string(index=False))

In [ ]:
# within-box shift against transport (Proposition 5): boxes released with source certificates
B = BOX.dropna(subset=['domain_auroc']).copy()
B['auroc_bin'] = pd.cut(B.domain_auroc, [-np.inf, 0.55, 0.60, 0.70, np.inf], labels=['<= 0.55', '0.55 to 0.60', '0.60 to 0.70', '> 0.70'])
for col in ('above_certificate', 'certificate_refuted', 'claim_refuted', 'above_adjusted_bound'):
    B[col] = B[col].astype(float)
DIAG = (B.groupby(['role', 'box_class', 'auroc_bin'], observed=True)
         .agg(boxes=('box', 'size'), median_ba=('ba_estimate', 'median'), above_certificate=('above_certificate', 'mean'),
              certificate_refuted=('certificate_refuted', 'mean'), claim_refuted=('claim_refuted', 'mean'),
              above_adjusted_bound=('above_adjusted_bound', 'mean')).reset_index())
DIAG['box_class'] = DIAG.box_class.map(KIND)
print('Boxes released with source certificates, by within-box domain AUROC (share of boxes):')
print(DIAG.round(3).to_string(index=False))
print(f'\nboxes analysed: {len(B):,} of {len(BOX):,} released (the rest have fewer than 50 subsampled records in a domain)')

## Figures

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedFormatter, FixedLocator, NullLocator, PercentFormatter
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False})
OKABE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00']
PANELS = [('temporal', 'Temporal: 2015 model, later years'), ('geographic', 'Census region held out'),
          ('state', 'States with most records')]
fig = plt.figure(figsize=(7.4, 6.1))
gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 0.36], hspace=0.24, wspace=0.3, left=0.1, right=0.98, top=0.95, bottom=0.035)
axes = np.empty((2, 3), dtype=object)
for j in range(3):
    axes[0, j] = fig.add_subplot(gs[0, j])
    axes[1, j] = fig.add_subplot(gs[1, j], sharex=axes[0, j])
for j, (setting, title) in enumerate(PANELS):
    tg = TGT[(TGT.setting == setting) & (TGT.role == 'target')].target.tolist()
    ref_setting = 'temporal' if setting == 'temporal' else 'geographic'
    for i, k in enumerate((0, 1)):
        ax = axes[i, j]
        rf = REC[(REC.setting == ref_setting) & (REC.role == 'reference') & (REC.kind == k)].groupby('m').coverage.mean()
        ax.plot(rf.index, rf.values, color='0.6', lw=1.6, marker='s', ms=2.5,
                label='No shift (source test split)' if setting == 'temporal' else 'No shift (mean of the 4 models)')
        for c, t in enumerate(tg):
            col = OKABE[c % len(OKABE)]
            s = REC[(REC.setting == setting) & (REC.target == t) & (REC.kind == k)].sort_values('m')
            ax.fill_between(s.m, s.coverage_q025, s.coverage_q975, color=col, alpha=0.13, lw=0)
            ax.plot(s.m, s.coverage, color=col, lw=1.2, marker='o', ms=2.8, label=t)
            tr = TRN[(TRN.setting == setting) & (TRN.target == t) & (TRN.kind == k)].coverage
            if len(tr):
                ax.axhline(float(tr.iloc[0]), color=col, lw=0.9, ls=(0, (4, 2)))
        ax.set_xscale('log')
        ax.xaxis.set_major_locator(FixedLocator(SIZES))
        ax.xaxis.set_major_formatter(FixedFormatter(['500', '1k', '2k', '5k', '10k'][:len(SIZES)]))
        ax.xaxis.set_minor_locator(NullLocator())
        ax.set_ylim(bottom=0)
        ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
        ax.grid(color='0.92', lw=0.6)
        if i == 0:
            ax.set_title(title, fontsize=8.5)
            ax.tick_params(labelbottom=False)
        if j == 0:
            ax.set_ylabel(f'{KIND[k].capitalize()} claims:\nshare of explained patients')
    lax = fig.add_subplot(gs[2, j]); lax.axis('off')
    h, l = axes[1, j].get_legend_handles_labels()
    lax.legend(h, l, loc='upper center', bbox_to_anchor=(0.5, 0.93), ncol=1, fontsize=6.5, frameon=False, borderaxespad=0)
axes[1, 1].set_xlabel('Labelled target records used for the certificates', fontsize=8.5)
fig.text(0.5, 0.0, f'Solid: certificates from labelled target records (mean of {DRAWS} draws; band: middle 95% of draws). '
         'Dashed: source certificates, no target labels.', ha='center', va='bottom', fontsize=6.5, color='0.3')
for ext in ('png', 'pdf'):
    fig.savefig(paths.FIGURES / f'fig_shift_recertification.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
SET_STYLE = {'temporal': ('#0072B2', 'Temporal: later years'), 'geographic': ('#D55E00', 'Census region held out'),
             'state': ('#009E73', 'States with most records')}
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.4))
for i, k in enumerate((0, 1)):
    ax = axes[i]
    d = BOX[(BOX.role == 'reference') & (BOX.box_class == k)].dropna(subset=['domain_auroc'])
    ax.scatter(d.domain_auroc, d.risk_target - d.risk_source, s=5, color='0.65', alpha=0.5, lw=0, label='No shift (source test split)')
    for setting, (col, lab) in SET_STYLE.items():
        d = BOX[(BOX.setting == setting) & (BOX.role == 'target') & (BOX.box_class == k)].dropna(subset=['domain_auroc'])
        ax.scatter(d.domain_auroc, d.risk_target - d.risk_source, s=5, color=col, alpha=0.5, lw=0, label=lab)
    ax.axhline(0, color='0.35', lw=0.6)
    ax.axvline(0.5, color='0.35', lw=0.6, ls=':')
    ax.set_title(f'{KIND[k].capitalize()} boxes released with source certificates', fontsize=8.5)
    ax.set_xlabel('Within-box domain AUROC (source against target records)')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=None))
    ax.grid(color='0.92', lw=0.6)
axes[0].set_ylabel('Label risk in the box: target minus source')
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=4, frameon=False, markerscale=2.5, fontsize=7, bbox_to_anchor=(0.5, 1.03))
fig.tight_layout(rect=(0, 0, 1, 0.93))
for ext in ('png', 'pdf'):
    fig.savefig(paths.FIGURES / f'fig_shift_box_diagnostics.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
TRN.to_csv(f'{paths.RESULTS}/13_shift.csv', index=False)
TGT.to_csv(f'{paths.RESULTS}/13_shift_targets.csv', index=False)
REC.to_csv(f'{paths.RESULTS}/13_shift_recertification.csv', index=False)
DRW.to_csv(f'{paths.RESULTS}/13_shift_recertification_draws.csv', index=False)
BOX.to_csv(f'{paths.RESULTS}/13_shift_boxes.csv', index=False, float_format='%.6g')
DIAG.to_csv(f'{paths.RESULTS}/13_shift_box_diagnostics.csv', index=False)
write_json(f'{paths.RESULTS}/13_shift.json', {
    'seed': SEED, 'frozen_config_sha256': FT.cfg_sha, 'brfss_parquet_sha256': REC02['parquet_sha256'],
    'targets': {'alpha': ALPHA, 'delta_1': DELTA1}, 'harmonised_set_H': H,
    'sizes': {'prior_patients': N_PRIOR, 'explained_per_target': N_EXPLAIN, 'explained_reference': N_REF,
              'explained_per_state': N_STATE, 'recertification_sizes': SIZES, 'draws': DRAWS,
              'min_checking_records': MIN_EVAL},
    'regions': REGIONS.to_dict('records'), 'records_without_region': int((~HAS_REGION).sum()),
    'states': [{'fips': f, 'name': sh.STATE_NAMES[f], 'region': sh.region_of([f])[0], 'records': int((STATE == f).sum())}
               for f in STATES],
    'models': MODELS, 'seconds_total': float(TGT.seconds.sum())})
print('written: results/13_shift.csv, 13_shift_targets.csv, 13_shift_recertification.csv, 13_shift_recertification_draws.csv, '
      '13_shift_boxes.csv, 13_shift_box_diagnostics.csv, 13_shift.json; figures/fig_shift_recertification, fig_shift_box_diagnostics')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "13: shift on BRFSS, temporal (2015 model with H to 2017-2023) and geographic (census regions, five states): transport, re-certification sample sizes, within-box shift, figures"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)